# Syria Population Raster Reprojection to Web Mercator

Overview

Reprojects the WorldPop 2026 population raster for Syria from WGS 84 (EPSG:4326) to Web Mercator (EPSG:3857).
The source values represent estimated population per source grid cell. Sum resampling is used during reprojection to retain the population represented by contributing source cells.
The output Raster is intended for alignment with web-map tiles that use Web Mercator. EPSG:3857 is not used for distance or area measurement in this workflow.
The workflow validates the source Raster, calculates the destination grid, performs the reprojection, compares population totals, saves the result as a GeoTIFF, and creates an interactive map.

WorldPop 2026のシリア人口Rasterを、WGS 84（EPSG:4326）からWeb Mercator（EPSG:3857）へ再投影します。
元データの値は、元グリッドセルごとの推計人口を表しています。対応する元セルの人口を保持するため、再投影には合計によるリサンプリングを使用します。
出力Rasterは、Web Mercatorを使用するWeb地図タイルとの座標系統一を目的とします。本処理では、EPSG:3857を距離・面積計測には使用しません。
元Rasterの検証、出力グリッドの計算、再投影、人口合計の比較、GeoTIFF保存、インタラクティブ地図の作成を行います。

Objectives

- Read and inspect the WorldPop 2026 population raster
- Validate the source CRS, dimensions, transform, bounds and NoData value
- Calculate an EPSG:3857 destination grid
- Reproject the population raster with sum resampling
- Compare population totals before and after reprojection
- Save the reprojected raster as a GeoTIFF
- Read back and validate the saved raster
- Create an interactive map with administrative boundaries, a legend and an information panel

- WorldPop 2026人口Rasterを読み込み、内容を確認する
- 元RasterのCRS、サイズ、Transform、範囲およびNoData値を検証する
- EPSG:3857の出力グリッドを計算する
- 合計によるリサンプリングを使用して人口Rasterを再投影する
- 再投影前後の人口合計を比較する
- 再投影結果をGeoTIFFとして保存する
- 保存したRasterを再読込して検証する
- 行政界、凡例および情報パネルを備えたインタラクティブ地図を作成する

Workflow

1. Define the source and output paths
2. Read and validate the source raster metadata
3. Read the source population values as a masked array
4. Calculate the EPSG:3857 destination grid
5. Reproject the Raster with sum resampling
6. Compare source and reprojected population totals
7. Save the reprojected Raster as a GeoTIFF
8. Read back and validate the saved Raster
9. Convert the output bounds for Folium display
10. Create an interactive map

1. 入力データと出力先のパスを定義する
2. 元Rasterのメタデータを読み込み、検証する
3. 元人口値をMaskedArrayとして読み込む
4. EPSG:3857の出力グリッドを計算する
5. 合計によるリサンプリングでRasterを再投影する
6. 元Rasterと再投影後Rasterの人口合計を比較する
7. 再投影結果をGeoTIFFとして保存する
8. 保存したRasterを再読込して検証する
9. Folium表示用に出力範囲を変換する
10. インタラクティブ地図を作成する

Data

Population raster data:
- worldpop_syria_2026.tif
- Source: WorldPop, open population data

Administrative boundary data:
- syr_admin0.geojson
- syr_admin1.geojson
- Source: HDX OCHA, Syria subnational administrative boundaries

Technologies

- Python
- Rasterio
- NumPy
- GeoPandas
- Folium
- Matplotlib
- GeoTIFF

In [ ]:
# 1
# Import the required libraries
# 必要なライブラリを読み込む

# File paths
# ファイルパス
from pathlib import Path

# Numerical processing
# 数値処理
import numpy as np

# Raster processing
# Raster処理
import rasterio
from rasterio.crs import CRS
from rasterio.enums import Resampling
from rasterio.transform import array_bounds
from rasterio.warp import (
    calculate_default_transform,
    reproject,
    transform_bounds,
)
from affine import Affine

# Vector processing
# Vector処理
import geopandas as gpd

# Web mapping and colour definition
# Web地図作成と色の定義
import folium
import matplotlib.colors as mcolors
from branca.element import Element

In [ ]:
# 2
# Define the source and output paths
# 入力データと出力先のパスを定義する

PROJECT_DIR = Path(
    "/Users/marisa/Syria_Humanitarian_Climate_Facts/"
    "01_PROJECTS/04_RASTER_OPERATIONS"
)

RASTER_DATA_DIR = Path(
    "/Users/marisa/Syria_Humanitarian_Climate_Facts/"
    "02_DATA/RASTER"
)

VECTOR_DATA_DIR = Path(
    "/Users/marisa/Syria_Humanitarian_Climate_Facts/"
    "02_DATA/VECTOR"
)

OUTPUT_DIR = PROJECT_DIR / "outputs"

population_raster_path = (
    RASTER_DATA_DIR / "worldpop_syria_2026.tif"
)

admin0_path = (
    VECTOR_DATA_DIR / "syr_admin0.geojson"
)

admin1_path = (
    VECTOR_DATA_DIR / "syr_admin1.geojson"
)

reprojected_raster_path = (
    OUTPUT_DIR
    / "worldpop_syria_2026_reprojected_3857.tif"
)

output_path = (
    PROJECT_DIR / "02_syria_raster_reprojection.html"
)

required_input_paths = {
    "Population raster": population_raster_path,
    "Country boundary": admin0_path,
    "Governorate boundaries": admin1_path,
}

missing_input_paths = [
    path
    for path in required_input_paths.values()
    if not path.exists()
]

if missing_input_paths:
    raise FileNotFoundError(
        "One or more required input files were not found:\n"
        + "\n".join(
            str(path)
            for path in missing_input_paths
        )
    )

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"Project directory: {PROJECT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Source raster: {population_raster_path}")

In [ ]:
# 3
# Read the source raster metadata
# 元Rasterのメタデータを読み込む

with rasterio.open(
    population_raster_path
) as src:

    source_crs = src.crs
    source_width = src.width
    source_height = src.height
    source_count = src.count
    source_transform = src.transform
    source_bounds = src.bounds
    source_resolution = src.res
    source_nodata = src.nodata
    source_dtype = src.dtypes[0]
    source_profile = src.profile.copy()

print(f"Source CRS: {source_crs}")

print(
    "Source dimensions: "
    f"{source_width:,} × {source_height:,}"
)

print(f"Source band count: {source_count}")
print(f"Source data type: {source_dtype}")
print(f"Source NoData: {source_nodata}")
print(f"Source resolution: {source_resolution}")
print(f"Source bounds: {source_bounds}")

print(
    f"Source transform:\n{source_transform}"
)

In [ ]:
# 4
# Validate the source raster metadata
# 元Rasterのメタデータを検証する

if source_crs is None:
    raise ValueError(
        "The source population raster has no defined CRS."
    )

if source_crs.to_epsg() != 4326:
    raise ValueError(
        "The source population raster was expected to use "
        f"EPSG:4326, but its CRS is {source_crs}."
    )

if source_count != 1:
    raise ValueError(
        "The source population raster was expected to "
        f"contain one band, but it contains {source_count}."
    )

if source_width <= 0 or source_height <= 0:
    raise ValueError(
        "The source raster dimensions are not valid."
    )

if source_nodata is None:
    raise ValueError(
        "The source population raster has no defined "
        "NoData value."
    )

if source_transform.a <= 0:
    raise ValueError(
        "The source raster has an invalid horizontal "
        "pixel size."
    )

if source_transform.e >= 0:
    raise ValueError(
        "The source raster was expected to be north-up."
    )

calculated_source_bounds = array_bounds(
    source_height,
    source_width,
    source_transform,
)

SOURCE_BOUNDS_TOLERANCE = 1e-9

if not np.allclose(
    calculated_source_bounds,
    tuple(source_bounds),
    atol=SOURCE_BOUNDS_TOLERANCE,
    rtol=0.0,
):
    raise ValueError(
        "The bounds calculated from the source transform "
        "do not match the stored source bounds."
    )

print(
    "Source raster metadata validation: passed"
)

print(
    "Calculated source bounds:",
    calculated_source_bounds,
)

In [ ]:
# 5
# Read and validate the source population values
# 元Rasterの人口値を読み込み、検証する

with rasterio.open(
    population_raster_path
) as src:

    source_population = src.read(
        1,
        masked=True,
    )

if not np.ma.isMaskedArray(
    source_population
):
    raise TypeError(
        "The source population raster was not read "
        "as a masked array."
    )

source_valid_pixel_count = int(
    source_population.count()
)

source_masked_pixel_count = int(
    source_population.size
    - source_valid_pixel_count
)

if source_valid_pixel_count == 0:
    raise ValueError(
        "The source population raster contains no "
        "valid pixels."
    )

source_min = float(
    source_population.min()
)

source_max = float(
    source_population.max()
)

source_mean = float(
    source_population.mean()
)

source_population_total = float(
    source_population.sum(
        dtype=np.float64
    )
)

if source_min < 0:
    raise ValueError(
        "The valid source population values contain "
        "negative values."
    )

if not np.isfinite(
    source_population_total
):
    raise ValueError(
        "The source population total is not finite."
    )

print(
    f"Source valid pixels: "
    f"{source_valid_pixel_count:,}"
)

print(
    f"Source masked pixels: "
    f"{source_masked_pixel_count:,}"
)

print(
    f"Source minimum value: {source_min:,.2f}"
)

print(
    f"Source maximum value: {source_max:,.2f}"
)

print(
    f"Source mean value: {source_mean:,.2f}"
)

print(
    "Source estimated population total: "
    f"{source_population_total:,.2f}"
)

In [ ]:
# 6
# Calculate the Web Mercator destination grid
# Web Mercatorの出力グリッドを計算する

TARGET_CRS = CRS.from_epsg(
    3857
)

(
    target_transform,
    target_width,
    target_height,
) = calculate_default_transform(
    source_crs,
    TARGET_CRS,
    source_width,
    source_height,
    *source_bounds,
)

target_resolution = (
    abs(target_transform.a),
    abs(target_transform.e),
)

target_bounds = array_bounds(
    target_height,
    target_width,
    target_transform,
)

transformed_source_bounds = transform_bounds(
    source_crs,
    TARGET_CRS,
    *source_bounds,
    densify_pts=21,
)

print(f"Target CRS: {TARGET_CRS}")

print(
    "Target dimensions: "
    f"{target_width:,} × {target_height:,}"
)

print(
    f"Target resolution: {target_resolution}"
)

print(
    f"Target bounds: {target_bounds}"
)

print(
    "Source bounds transformed to target CRS:",
    transformed_source_bounds,
)

print(
    f"Target transform:\n{target_transform}"
)

In [ ]:
# 7
# Validate the Web Mercator destination grid
# Web Mercatorの出力グリッドを検証する

if TARGET_CRS.to_epsg() != 3857:
    raise ValueError(
        "The target CRS was expected to be EPSG:3857."
    )

if not TARGET_CRS.is_projected:
    raise ValueError(
        "The target CRS was expected to be projected."
    )

if target_width <= 0 or target_height <= 0:
    raise ValueError(
        "The target raster dimensions are not valid."
    )

if target_transform.a <= 0:
    raise ValueError(
        "The target raster has an invalid horizontal "
        "pixel size."
    )

if target_transform.e >= 0:
    raise ValueError(
        "The target raster was expected to be north-up."
    )

if not np.isclose(
    target_resolution[0],
    target_resolution[1],
    rtol=1e-6,
    atol=0.0,
):
    raise ValueError(
        "The target raster pixels are not square."
    )

TARGET_BOUNDS_TOLERANCE = (
    max(target_resolution)
    * 2
)

if not np.allclose(
    target_bounds,
    transformed_source_bounds,
    atol=TARGET_BOUNDS_TOLERANCE,
    rtol=0.0,
):
    raise ValueError(
        "The target raster bounds do not correspond "
        "to the transformed source bounds."
    )

print(
    "Target grid validation: passed"
)

print(
    "Target pixel size in metres: "
    f"{target_resolution[0]:,.2f}"
)

print(
    "Target-bounds tolerance in metres: "
    f"{TARGET_BOUNDS_TOLERANCE:,.2f}"
)

In [ ]:
# 8
# Reproject the population raster with sum resampling
# 合計によるリサンプリングで人口Rasterを再投影する

# Release the full source array before allocating
# the destination array.
# 出力配列を確保する前に元Raster配列をメモリから解放する
if "source_population" in globals():
    del source_population

reprojected_output_data = np.full(
    (
        target_height,
        target_width,
    ),
    source_nodata,
    dtype=np.float32,
)

with rasterio.open(
    population_raster_path
) as src:

    reproject(
        source=rasterio.band(
            src,
            1,
        ),
        destination=reprojected_output_data,
        src_transform=source_transform,
        src_crs=source_crs,
        src_nodata=source_nodata,
        dst_transform=target_transform,
        dst_crs=TARGET_CRS,
        dst_nodata=source_nodata,
        resampling=Resampling.sum,
        init_dest_nodata=True,
        num_threads=2,
    )

reprojected_population = np.ma.masked_equal(
    reprojected_output_data,
    source_nodata,
)

if reprojected_population.shape != (
    target_height,
    target_width,
):
    raise ValueError(
        "The reprojected raster dimensions do not match "
        "the target grid."
    )

if not np.ma.isMaskedArray(
    reprojected_population
):
    raise TypeError(
        "The reprojected raster was not created as a "
        "masked array."
    )

reprojected_valid_pixel_count = int(
    reprojected_population.count()
)

if reprojected_valid_pixel_count == 0:
    raise ValueError(
        "The reprojected population raster contains "
        "no valid pixels."
    )

reprojected_min = float(
    reprojected_population.min()
)

reprojected_max = float(
    reprojected_population.max()
)

reprojected_mean = float(
    reprojected_population.mean()
)

reprojected_population_total = float(
    reprojected_population.sum(
        dtype=np.float64
    )
)

if reprojected_min < 0:
    raise ValueError(
        "The valid reprojected population values "
        "contain negative values."
    )

if not np.isfinite(
    reprojected_population_total
):
    raise ValueError(
        "The reprojected population total is not finite."
    )

print(
    "Reprojection method: "
    "Rasterio warp with Resampling.sum"
)

print(
    "Reprojected raster shape: "
    f"{reprojected_population.shape}"
)

print(
    "Reprojected valid pixels: "
    f"{reprojected_valid_pixel_count:,}"
)

print(
    f"Reprojected minimum value: "
    f"{reprojected_min:,.2f}"
)

print(
    f"Reprojected maximum value: "
    f"{reprojected_max:,.2f}"
)

print(
    f"Reprojected mean value: "
    f"{reprojected_mean:,.2f}"
)

print(
    "Reprojected estimated population total: "
    f"{reprojected_population_total:,.2f}"
)

In [ ]:
# 9
# Compare population totals before and after reprojection
# 再投影前後の人口合計を比較する

if source_population_total <= 0:
    raise ValueError(
        "The source population total must be greater "
        "than zero."
    )

population_difference = (
    reprojected_population_total
    - source_population_total
)

absolute_population_difference = abs(
    population_difference
)

population_difference_percent = (
    absolute_population_difference
    / source_population_total
    * 100
)

population_preservation_percent = (
    reprojected_population_total
    / source_population_total
    * 100
)

MAX_POPULATION_DIFFERENCE_PERCENT = 0.1

if (
    population_difference_percent
    > MAX_POPULATION_DIFFERENCE_PERCENT
):
    raise ValueError(
        "The population difference after reprojection "
        "exceeds the permitted threshold. "
        f"Difference: {population_difference_percent:.6f}%"
    )

print(
    "Source estimated population total: "
    f"{source_population_total:,.2f}"
)

print(
    "Reprojected estimated population total: "
    f"{reprojected_population_total:,.2f}"
)

print(
    "Population difference: "
    f"{population_difference:,.2f}"
)

print(
    "Absolute difference percentage: "
    f"{population_difference_percent:.6f}%"
)

print(
    "Population preservation percentage: "
    f"{population_preservation_percent:.6f}%"
)

In [ ]:
# 10
# Prepare the reprojected raster metadata
# 再投影後Rasterのメタデータを準備する

OUTPUT_DTYPE = "float32"

reprojected_output_array = (
    reprojected_population
    .filled(source_nodata)
    .astype(OUTPUT_DTYPE)
)

output_profile = source_profile.copy()

output_profile.update(
    driver="GTiff",
    count=1,
    width=target_width,
    height=target_height,
    crs=TARGET_CRS,
    transform=target_transform,
    dtype=OUTPUT_DTYPE,
    nodata=source_nodata,
    compress="LZW",
    predictor=3,
    tiled=True,
)

if reprojected_output_array.shape != (
    target_height,
    target_width,
):
    raise ValueError(
        "The output array dimensions do not match "
        "the output metadata."
    )

if reprojected_output_array.dtype != np.dtype(
    OUTPUT_DTYPE
):
    raise TypeError(
        "The output array does not use the defined "
        "data type."
    )

print(
    f"Output array shape: "
    f"{reprojected_output_array.shape}"
)

print(
    f"Output data type: "
    f"{reprojected_output_array.dtype}"
)

print(
    f"Output CRS: {output_profile['crs']}"
)

print(
    f"Output NoData: {output_profile['nodata']}"
)

print(
    f"Output compression: {output_profile['compress']}"
)

In [ ]:
# 11
# Save the reprojected population raster as a GeoTIFF
# 再投影した人口RasterをGeoTIFFとして保存する

with rasterio.open(
    reprojected_raster_path,
    "w",
    **output_profile,
) as dst:

    dst.write(
        reprojected_output_array,
        1,
    )

print(
    f"Reprojected raster saved to: "
    f"{reprojected_raster_path}"
)

print(
    "Output file size: "
    f"{reprojected_raster_path.stat().st_size / 1024**2:,.2f} MB"
)

In [ ]:
# 12
# Read back the saved reprojected raster
# 保存した再投影Rasterを再読込する

with rasterio.open(
    reprojected_raster_path
) as src:

    saved_population = src.read(
        1,
        masked=True,
    )

    saved_crs = src.crs
    saved_width = src.width
    saved_height = src.height
    saved_count = src.count
    saved_transform = src.transform
    saved_bounds = src.bounds
    saved_resolution = src.res
    saved_nodata = src.nodata
    saved_dtype = src.dtypes[0]
    saved_compression = src.compression

saved_valid_pixel_count = int(
    saved_population.count()
)

saved_population_total = float(
    saved_population.sum(
        dtype=np.float64
    )
)

print(f"Saved CRS: {saved_crs}")

print(
    "Saved dimensions: "
    f"{saved_width:,} × {saved_height:,}"
)

print(f"Saved band count: {saved_count}")
print(f"Saved data type: {saved_dtype}")
print(f"Saved NoData: {saved_nodata}")
print(f"Saved resolution: {saved_resolution}")
print(f"Saved bounds: {saved_bounds}")
print(f"Saved compression: {saved_compression}")

print(
    "Saved valid pixels: "
    f"{saved_valid_pixel_count:,}"
)

print(
    "Saved estimated population total: "
    f"{saved_population_total:,.2f}"
)

In [ ]:
# 13
# Validate the saved reprojected raster
# 保存した再投影Rasterを検証する

if saved_crs != TARGET_CRS:
    raise ValueError(
        "The saved raster does not use the target CRS."
    )

if saved_width != target_width:
    raise ValueError(
        "The saved raster width does not match the "
        "target width."
    )

if saved_height != target_height:
    raise ValueError(
        "The saved raster height does not match the "
        "target height."
    )

if saved_count != 1:
    raise ValueError(
        "The saved raster does not contain one band."
    )

if saved_transform != target_transform:
    raise ValueError(
        "The saved raster does not retain the target "
        "transform."
    )

if not np.allclose(
    tuple(saved_bounds),
    target_bounds,
    atol=1e-6,
    rtol=0.0,
):
    raise ValueError(
        "The saved raster bounds do not match the "
        "target bounds."
    )

if saved_nodata != source_nodata:
    raise ValueError(
        "The saved raster does not retain the source "
        "NoData value."
    )

if saved_dtype != OUTPUT_DTYPE:
    raise ValueError(
        "The saved raster does not use the defined "
        "output data type."
    )

if saved_population.shape != (
    target_height,
    target_width,
):
    raise ValueError(
        "The saved raster array has unexpected "
        "dimensions."
    )

saved_total_difference = abs(
    saved_population_total
    - reprojected_population_total
)

saved_total_difference_percent = (
    saved_total_difference
    / reprojected_population_total
    * 100
)

if (
    saved_total_difference_percent
    > MAX_POPULATION_DIFFERENCE_PERCENT
):
    raise ValueError(
        "The saved raster population total differs "
        "from the in-memory reprojected total."
    )

print(
    "Saved raster validation: passed"
)

print(
    "Saved-total difference percentage: "
    f"{saved_total_difference_percent:.6f}%"
)

In [ ]:
# 14
# Read and validate the administrative boundary datasets
# 行政界データを読み込み、検証する

admin0 = gpd.read_file(
    admin0_path
)

admin1 = gpd.read_file(
    admin1_path
)

administrative_datasets = {
    "Country boundaries": admin0,
    "Governorate boundaries": admin1,
}

for dataset_name, dataset in (
    administrative_datasets.items()
):

    if dataset.crs is None:
        raise ValueError(
            f"{dataset_name} has no defined CRS."
        )

    if dataset.crs.to_epsg() != 4326:
        raise ValueError(
            f"{dataset_name} was expected to use "
            f"EPSG:4326, but its CRS is {dataset.crs}."
        )

    if dataset.empty:
        raise ValueError(
            f"{dataset_name} contains no features."
        )

    if dataset.geometry.isna().any():
        raise ValueError(
            f"{dataset_name} contains missing geometries."
        )

    if dataset.geometry.is_empty.any():
        raise ValueError(
            f"{dataset_name} contains empty geometries."
        )

    if not dataset.geometry.is_valid.all():
        raise ValueError(
            f"{dataset_name} contains invalid geometries."
        )

    print(
        f"{dataset_name}: "
        f"{len(dataset):,} features, {dataset.crs}"
    )

In [ ]:
# 15
# Prepare an aggregated copy for web display
# Web表示用の集約コピーを作成する

DISPLAY_MAX_DIMENSION = 1000

display_scale = (
    max(
        saved_width,
        saved_height,
    )
    / DISPLAY_MAX_DIMENSION
)

display_width = int(
    round(
        saved_width
        / display_scale
    )
)

display_height = int(
    round(
        saved_height
        / display_scale
    )
)

display_transform = (
    saved_transform
    * Affine.scale(
        saved_width / display_width,
        saved_height / display_height,
    )
)

display_resolution_metres = (
    abs(display_transform.a),
    abs(display_transform.e),
)

display_output_data = np.full(
    (
        display_height,
        display_width,
    ),
    saved_nodata,
    dtype=np.float32,
)

with rasterio.open(
    reprojected_raster_path
) as src:

    reproject(
        source=rasterio.band(
            src,
            1,
        ),
        destination=display_output_data,
        src_transform=saved_transform,
        src_crs=saved_crs,
        src_nodata=saved_nodata,
        dst_transform=display_transform,
        dst_crs=saved_crs,
        dst_nodata=saved_nodata,
        resampling=Resampling.sum,
        init_dest_nodata=True,
        num_threads=2,
    )

display_population_masked = np.ma.masked_equal(
    display_output_data,
    saved_nodata,
)

display_population_total = float(
    display_population_masked.sum(
        dtype=np.float64
    )
)

display_total_difference_percent = (
    abs(
        display_population_total
        - saved_population_total
    )
    / saved_population_total
    * 100
)

if (
    display_total_difference_percent
    > MAX_POPULATION_DIFFERENCE_PERCENT
):
    raise ValueError(
        "The web-display population total differs "
        "from the saved raster total."
    )

display_bounds_3857 = array_bounds(
    display_height,
    display_width,
    display_transform,
)

display_bounds_4326 = transform_bounds(
    saved_crs,
    CRS.from_epsg(4326),
    *display_bounds_3857,
    densify_pts=21,
)

west, south, east, north = (
    display_bounds_4326
)

folium_bounds = [
    [
        south,
        west,
    ],
    [
        north,
        east,
    ],
]

print(
    "Web-display dimensions: "
    f"{display_width:,} × {display_height:,}"
)

print(
    "Web-display resolution in metres: "
    f"{display_resolution_metres}"
)

print(
    "Web-display population preservation: "
    f"{100 - display_total_difference_percent:.6f}%"
)

print(
    f"Folium bounds: {folium_bounds}"
)

In [ ]:
# 16
# Prepare the display values and colour map
# 表示値とカラーマップを準備する

display_mask = np.ma.getmaskarray(
    display_population_masked
)

display_valid_values = (
    display_population_masked.compressed()
)

DISPLAY_PERCENTILE = 99.0

display_upper_value = float(
    np.percentile(
        display_valid_values,
        DISPLAY_PERCENTILE,
    )
)

if display_upper_value <= 0:
    raise ValueError(
        "The display upper value must be greater "
        "than zero."
    )

clipped_display_population = np.clip(
    display_population_masked.filled(np.nan),
    0,
    display_upper_value,
)

display_population = np.full(
    display_population_masked.shape,
    np.nan,
    dtype=np.float32,
)

valid_display_pixels = (
    ~display_mask
    & np.isfinite(
        clipped_display_population
    )
)

display_population[
    valid_display_pixels
] = (
    np.log1p(
        clipped_display_population[
            valid_display_pixels
        ]
    )
    / np.log1p(
        display_upper_value
    )
)

colour_stops = [
    (0.00, "#7bb5a000"),
    (0.20, "#7bb5a0ff"),
    (0.40, "#2e9166ff"),
    (0.60, "#226c4cff"),
    (0.80, "#184d36ff"),
    (1.00, "#0f3122ff"),
]

population_cmap = (
    mcolors.LinearSegmentedColormap.from_list(
        "SyriaPopulationReprojection",
        colour_stops,
    )
)

population_cmap.set_bad(
    color=(
        0,
        0,
        0,
        0,
    )
)

legend_positions = np.array(
    [
        0.00,
        0.25,
        0.50,
        0.75,
        1.00,
    ]
)

legend_values = np.expm1(
    legend_positions
    * np.log1p(
        display_upper_value
    )
)

print(
    f"Display upper value: "
    f"{display_upper_value:,.2f}"
)

print(
    "Legend values:",
    [
        round(value)
        for value in legend_values
    ],
)

In [ ]:
# 17
# Create the no-label base map and set the initial extent
# 地名表記のないベースマップを作成し、初期表示範囲を設定する

m = folium.Map(
    location=[
        34.8,
        38.5,
    ],
    zoom_start=6,
    tiles=None,
    control_scale=True,
    prefer_canvas=True,
)

folium.TileLayer(
    tiles=(
        "https://{s}.basemaps.cartocdn.com/"
        "light_nolabels/{z}/{x}/{y}{r}.png"
    ),
    attr=(
        '&copy; '
        '<a href="https://www.openstreetmap.org/copyright">'
        'OpenStreetMap</a> contributors '
        '&copy; '
        '<a href="https://carto.com/attributions">'
        'CARTO</a>'
    ),
    name="CARTO Light — No Labels",
    control=True,
    show=True,
).add_to(
    m
)

m.fit_bounds(
    folium_bounds,
    padding=[
        25,
        25,
    ],
)

In [ ]:
# 18
# Add the reprojected population raster layer
# 再投影した人口Rasterレイヤーを追加する

population_raster_layer = folium.FeatureGroup(
    name="Reprojected Population (EPSG:3857)",
    show=True,
)

folium.raster_layers.ImageOverlay(
    image=display_population,
    bounds=folium_bounds,
    colormap=population_cmap,
    opacity=0.82,
    name="Reprojected Population",
    interactive=True,
    zindex=1,
).add_to(
    population_raster_layer
)

population_raster_layer.add_to(
    m
)

In [ ]:
# 19
# Add the country and governorate boundaries
# 国境および県境レイヤーを追加する

country_boundary_layer = folium.FeatureGroup(
    name="Country Boundary",
    show=True,
)

folium.GeoJson(
    data=admin0[
        [
            "adm0_name",
            "adm0_pcode",
            "geometry",
        ]
    ].to_json(),
    style_function=lambda feature: {
        "fillColor": "transparent",
        "color": "#202020",
        "weight": 2.8,
        "fillOpacity": 0.0,
        "opacity": 0.95,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "adm0_name",
            "adm0_pcode",
        ],
        aliases=[
            "Country:",
            "Pcode:",
        ],
        sticky=False,
    ),
).add_to(
    country_boundary_layer
)

country_boundary_layer.add_to(
    m
)

governorate_boundary_layer = folium.FeatureGroup(
    name="Governorate Boundaries",
    show=True,
)

folium.GeoJson(
    data=admin1[
        [
            "adm1_name",
            "adm1_pcode",
            "geometry",
        ]
    ].to_json(),
    style_function=lambda feature: {
        "fillColor": "transparent",
        "color": "#666666",
        "weight": 1.0,
        "fillOpacity": 0.0,
        "opacity": 0.85,
    },
    highlight_function=lambda feature: {
        "color": "#111111",
        "weight": 2.0,
        "opacity": 1.0,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "adm1_name",
            "adm1_pcode",
        ],
        aliases=[
            "Governorate:",
            "Pcode:",
        ],
        sticky=False,
    ),
).add_to(
    governorate_boundary_layer
)

governorate_boundary_layer.add_to(
    m
)

In [ ]:
# 20
# Add governorate and neighbouring-country labels
# 県名および周辺国名ラベルを追加する

if not admin1[
    [
        "center_lat",
        "center_lon",
    ]
].notna().all().all():
    raise ValueError(
        "One or more governorate label coordinates are missing."
    )

governorate_label_layer = folium.FeatureGroup(
    name="Governorate Labels",
    show=True,
)

for _, governorate in admin1.iterrows():

    folium.Marker(
        location=[
            governorate["center_lat"],
            governorate["center_lon"],
        ],
        icon=folium.DivIcon(
            icon_size=(
                130,
                24,
            ),
            icon_anchor=(
                65,
                12,
            ),
            html=f"""
            <div style="
                font-size: 12px;
                font-weight: 700;
                color: #202020;
                text-align: center;
                white-space: nowrap;
                text-shadow:
                    -1px -1px 0 #ffffff,
                     1px -1px 0 #ffffff,
                    -1px  1px 0 #ffffff,
                     1px  1px 0 #ffffff;
            ">
                {governorate["adm1_name"]}
            </div>
            """,
        ),
    ).add_to(
        governorate_label_layer
    )

governorate_label_layer.add_to(
    m
)

neighbour_label_layer = folium.FeatureGroup(
    name="Neighbour Labels",
    show=True,
)

neighbour_labels = {
    "TÜRKİYE": [37.75, 38.1],
    "IRAQ": [34.5, 42.75],
    "LEBANON": [33.8, 35.5],
    "JORDAN": [31.85, 37.2],
}

for country_name, coordinates in (
    neighbour_labels.items()
):

    folium.Marker(
        location=coordinates,
        icon=folium.DivIcon(
            icon_size=(
                150,
                30,
            ),
            icon_anchor=(
                75,
                15,
            ),
            html=f"""
            <div style="
                font-size: 18px;
                font-weight: 700;
                color: #666666;
                text-align: center;
                white-space: nowrap;
                text-shadow:
                    -1px -1px 0 #ffffff,
                     1px -1px 0 #ffffff,
                    -1px  1px 0 #ffffff,
                     1px  1px 0 #ffffff;
            ">
                {country_name}
            </div>
            """,
        ),
    ).add_to(
        neighbour_label_layer
    )

neighbour_label_layer.add_to(
    m
)

In [ ]:
# 21
# Add the map information and source panel
# 地図の説明、処理方法および出典を追加する

information_panel_html = f"""
<div style="
    position: fixed;
    top: 20px;
    left: 50px;
    width: 440px;
    min-height: 235px;
    background-color: rgba(255, 255, 255, 0.94);
    color: #222222;
    z-index: 9000;
    font-size: 14px;
    border: 1px solid #555555;
    border-radius: 8px;
    padding: 12px;
    box-shadow: 0 0 15px rgba(0, 0, 0, 0.25);
">
    <b style="font-size: 16px;">
        Syria
    </b>
    <br>

    <span style="
        color: #226c4c;
        font-weight: bold;
    ">
        Population Raster Reprojection to Web Mercator
    </span>

    <small style="
        display: block;
        margin-top: 7px;
        line-height: 1.35;
        color: #333333;
    ">
        The WorldPop 2026 population Raster was
        reprojected from WGS 84 (EPSG:4326) to
        Web Mercator (EPSG:3857).
        Sum resampling was used because source values
        represent estimated population per grid cell.
        The output is intended for web-map alignment,
        not for distance or area measurement.
        The displayed layer is an aggregated copy;
        the saved GeoTIFF retains the target grid.
    </small>

    <div style="
        margin-top: 10px;
        padding-top: 7px;
        font-size: 11px;
        line-height: 1.35;
        color: #555555;
        border-top: 1px solid #aaaaaa;
    ">
        Source:
        <a
            href="https://www.worldpop.org/"
            target="_blank"
            style="
                color: #226c4c;
                text-decoration: none;
                font-weight: bold;
            "
        >
            WorldPop 2026
        </a><br>

        Source CRS:
        <b>EPSG:4326</b><br>

        Output CRS:
        <b>EPSG:3857</b><br>

        Output dimensions:
        <b>{saved_width:,} × {saved_height:,}</b><br>

        Output pixel size:
        <b>{saved_resolution[0]:,.2f} m</b><br>

        Population preservation:
        <b>{population_preservation_percent:.6f}%</b><br>

        Method:
        Raster Warp / Sum Resampling /
        GeoTIFF Export
    </div>
</div>
"""

m.get_root().html.add_child(
    Element(
        information_panel_html
    )
)

In [ ]:
# 22
# Add the population legend and layer control
# 人口凡例とレイヤー切替コントロールを追加する

legend_labels = [
    f"{value:,.0f}"
    for value in legend_values
]

legend_html = f"""
<div style="
    position: fixed;
    right: 35px;
    bottom: 25px;
    width: 350px;
    background-color: rgba(255, 255, 255, 0.94);
    color: #222222;
    z-index: 9000;
    font-size: 12px;
    border: 1px solid #555555;
    border-radius: 8px;
    padding: 12px;
    box-shadow: 0 0 12px rgba(0, 0, 0, 0.22);
">
    <b style="font-size: 14px;">
        Estimated population per display grid cell
    </b>

    <div style="
        width: 100%;
        height: 14px;
        margin-top: 10px;
        background: linear-gradient(
            to right,
            rgba(123, 181, 160, 0),
            #7bb5a0 20%,
            #2e9166 40%,
            #226c4c 60%,
            #184d36 80%,
            #0f3122 100%
        );
        border: 1px solid #777777;
    "></div>

    <div style="
        display: flex;
        justify-content: space-between;
        margin-top: 3px;
        font-size: 10px;
    ">
        <span>{legend_labels[0]}</span>
        <span>{legend_labels[1]}</span>
        <span>{legend_labels[2]}</span>
        <span>{legend_labels[3]}</span>
        <span>{legend_labels[4]}</span>
    </div>

    <div style="
        margin-top: 10px;
        padding-top: 7px;
        border-top: 1px solid #aaaaaa;
        font-size: 11px;
        line-height: 1.4;
        color: #555555;
    ">
        Display scaling: logarithmic<br>
        Display cap: 99th percentile<br>
        Saved Raster resolution:
        {saved_resolution[0]:,.2f} m<br>
        Saved Raster CRS: EPSG:3857
    </div>
</div>
"""

m.get_root().html.add_child(
    Element(
        legend_html
    )
)

folium.LayerControl(
    position="topright",
    collapsed=False,
).add_to(
    m
)

In [ ]:
# 23
# Save and display the interactive map
# インタラクティブ地図を保存し、Notebook上に表示する

m.save(
    output_path
)

print(
    f"Map saved to: {output_path}"
)

m